# 03 — Local Research Pipeline

## Data-preparation stage

This is the single local research notebook for work after environment setup and data download. It validates the downloaded CulturaX artifacts, creates deterministic train/validation/test streaming manifests, trains the matched BPE and probabilistic tokenizers, then pretrains and saves their two randomly initialized XLM-R-style MLMs.

The 45 GB CulturaX text is **not** expanded into another local Arrow or text copy. The plans retain the validated Parquet shard references and apply the same configured normalization and deterministic document-level train/validation/test split at iteration time. This preserves disk space while keeping training fully offline and reproducible.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    configured_root = os.environ.get("PROJECT_ROOT")
    starting_points = [Path(configured_root)] if configured_root else [Path.cwd()]
    for starting_point in starting_points:
        resolved_start = starting_point.expanduser().resolve()
        for candidate in (resolved_start, *resolved_start.parents):
            if (candidate / "configs" / "config.yaml").is_file() and (candidate / "instructions.md").is_file():
                return candidate
    raise FileNotFoundError("Could not locate PROJECT_ROOT. Launch Jupyter from the repository root or set PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## 1. Load configuration and set the deterministic seed

All preparation behavior comes from `configs/config.yaml`. `train` creates missing processed artifacts, `resume` validates and reuses valid artifacts, and `evaluate` validates the prepared artifacts without creating a new corpus plan or downstream dataset.

In [ ]:
import json
import random

import numpy as np
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

required_sections = {"runtime", "checkpointing", "paths", "data_download", "data_preparation", "training"}
missing_sections = required_sections.difference(config)
if missing_sections:
    raise KeyError(f"Missing configuration sections: {sorted(missing_sections)}. Run 01_environment_setup.ipynb first.")

SEED = int(config["training"]["seed"])
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
except ImportError:
    torch = None

print(f"Runtime mode: {config['runtime']['mode']}")
print(f"Preparation seed: {SEED}")
print(f"Corpus storage mode: {config['data_preparation']['corpus_storage']['mode']}")

## 2. Validate downloaded data

The preparation code verifies every completed CulturaX Parquet shard and every GLueCoS `Dataset.save_to_disk()` artifact before it creates or reuses an output. It also confirms that the three CulturaX source subsets remain balanced within the configured byte tolerance.

In [ ]:
from src.data.pipeline import run_data_preparation

preparation_result = run_data_preparation(PROJECT_ROOT, config, include_downstream=True)
print(json.dumps(preparation_result, indent=2, ensure_ascii=False))

## 3. Inspect training-ready corpus plans

The tokenizer and MLM plans point to the local CulturaX shards, retain the configured NFKC/whitespace policy, and use the same SHA-256-based deterministic train/validation/test split. Their source-byte balance is intentional; exact post-tokenization token-count balancing is recorded after baseline BPE training.

In [ ]:
PROCESSED_ROOT = PROJECT_ROOT / config["paths"]["processed_data"]
for purpose in ("tokenizer_training", "mlm_training"):
    plan_path = PROCESSED_ROOT / purpose / "manifest.json"
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    byte_summary = {entry["language"]: entry["source_text_bytes"] for entry in plan["languages"]}
    print(f"{purpose}: {byte_summary}")
    print(f"  final token balance: {plan['balancing']['final_token_count_status']}")

## 4. Smoke-test local streaming

This reads only three normalized records—one at a time in round-robin language order—from the tokenizer plan. It does not download, decompress, or materialize the full corpus.

In [ ]:
from itertools import islice

from src.data.pipeline import iter_prepared_corpus

tokenizer_plan_path = PROCESSED_ROOT / "tokenizer_training" / "manifest.json"
sample_records = list(islice(iter_prepared_corpus(PROJECT_ROOT, tokenizer_plan_path, split="train", balanced=True), 3))
for record in sample_records:
    print({"language": record["language"], "characters": len(record["text"]), "preview": record["text"][:120]})

## 5. Checkpoint handoff

The preparation manifest is the checkpoint for this stage. The next notebook sections will train the baseline BPE tokenizer from `data/processed/tokenizer_training/manifest.json`, then use the same streaming preprocessing plan for MLM. No model training occurs in this data-preparation stage.

In [ ]:
PIPELINE_MANIFEST_PATH = PROJECT_ROOT / config["checkpointing"]["manifest_path"]
pipeline_manifest = json.loads(PIPELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
for stage_name, stage in pipeline_manifest.get("stages", {}).items():
    print(f"{stage_name}: {stage['status']}")
print(f"Dataset provenance: {PROJECT_ROOT / 'experiments' / 'research_pipeline' / 'dataset_info.json'}")

## 6. Train the CulturaX-only language prior and save the matched tokenizer pair

The lightweight character n-gram language prior is trained only on CulturaX train texts and language IDs; it never sees downstream labels. Both tokenizers then consume only the same fixed CulturaX **train** split, in the same round-robin English/Spanish/Hindi document order with equal language weights and the same normalized-text budget. The BPE tokenizer is the control artifact. The proposed artifact is a freshly trained Unigram tokenizer that retains its learned segmentation scores and candidate settings.

This cell can take a substantial time because the default budget is the complete 90% training split. It is restart-safe: a completed artifact with a matching corpus/configuration identity is reused; an incomplete artifact is never overwritten.

In [ ]:
from src.language.language_classifier import CulturaXLanguagePriorClassifier, train_culturax_language_prior_classifier
from src.tokenizer.balance import measure_bpe_token_balance
from src.tokenizer.bpe import train_bpe_tokenizer, validate_bpe_artifact
from src.tokenizer.probabilistic import train_probabilistic_tokenizer, validate_probabilistic_artifact

TOKENIZER_OUTPUT_ROOT = PROJECT_ROOT / config['paths']['tokenizers']
BPE_TOKENIZER_DIR = TOKENIZER_OUTPUT_ROOT / 'bpe'
PROBABILISTIC_TOKENIZER_DIR = TOKENIZER_OUTPUT_ROOT / 'probabilistic'
LANGUAGE_CLASSIFIER_DIR = PROJECT_ROOT / config['paths']['language_classifier']
MLM_PLAN_PATH = PROCESSED_ROOT / 'mlm_training' / 'manifest.json'
BPE_BALANCE_PATH = PROJECT_ROOT / config['paths']['results'] / 'tokenizer_balance' / 'bpe_token_balance.json'
MAX_SEQUENCE_LENGTH = int(config['model']['max_sequence_length'])

if config['runtime']['mode'] == 'evaluate':
    language_classifier_info = json.loads((LANGUAGE_CLASSIFIER_DIR / 'training_metadata.json').read_text(encoding='utf-8'))
    CulturaXLanguagePriorClassifier.load(LANGUAGE_CLASSIFIER_DIR)
    bpe_tokenizer_info = validate_bpe_artifact(BPE_TOKENIZER_DIR)
    probabilistic_tokenizer_info = validate_probabilistic_artifact(PROBABILISTIC_TOKENIZER_DIR)
else:
    language_classifier_info = train_culturax_language_prior_classifier(
        PROJECT_ROOT, MLM_PLAN_PATH, config['probabilistic_tokenizer']['language_classifier'], LANGUAGE_CLASSIFIER_DIR
    )
    bpe_tokenizer_info = train_bpe_tokenizer(
        PROJECT_ROOT, tokenizer_plan_path, config['tokenizer'], MAX_SEQUENCE_LENGTH, BPE_TOKENIZER_DIR
    )
    bpe_token_balance = measure_bpe_token_balance(
        PROJECT_ROOT, tokenizer_plan_path, config['tokenizer'], BPE_TOKENIZER_DIR, BPE_BALANCE_PATH
    )
    probabilistic_tokenizer_info = train_probabilistic_tokenizer(
        PROJECT_ROOT, tokenizer_plan_path, config['probabilistic_tokenizer'], MAX_SEQUENCE_LENGTH, PROBABILISTIC_TOKENIZER_DIR
    )

if bpe_tokenizer_info['vocab_size'] != probabilistic_tokenizer_info['vocab_size']:
    raise ValueError('Comparison invalid: tokenizer vocabulary sizes differ.')
print('Language-classifier examples:', language_classifier_info['examples_by_language'])
print('BPE vocabulary:', bpe_tokenizer_info['vocab_size'])
if BPE_BALANCE_PATH.is_file():
    bpe_token_balance = json.loads(BPE_BALANCE_PATH.read_text(encoding='utf-8'))
    print('BPE maximum language token-share deviation:', bpe_token_balance['max_absolute_share_deviation_from_equal'])
print('Probabilistic vocabulary:', probabilistic_tokenizer_info['vocab_size'])

## 7. Validate candidate alignment and tokenizer diagnostics

This bounded CulturaX validation sample validates original-character candidate spans and saves comparable tokenization diagnostics before MLM pretraining. It uses no downstream labels.

In [ ]:
from src.tokenizer.comparison import run_tokenizer_diagnostics

TOKENIZER_DIAGNOSTICS_PATH = PROJECT_ROOT / config['paths']['results'] / 'tokenizer_diagnostics.json'
tokenizer_diagnostics = run_tokenizer_diagnostics(
    PROJECT_ROOT, tokenizer_plan_path, BPE_TOKENIZER_DIR, PROBABILISTIC_TOKENIZER_DIR, LANGUAGE_CLASSIFIER_DIR,
    MAX_SEQUENCE_LENGTH, int(config['tokenizer_diagnostics']['validation_examples_per_language']), TOKENIZER_DIAGNOSTICS_PATH
)
print(json.dumps(tokenizer_diagnostics, indent=2, ensure_ascii=False))

## 8. Pretrain and save the two matched XLM-R-style MLMs

Each group receives a new random XLM-R-base-style model. The configured model architecture, random seed, CulturaX train/validation split, multilingual document schedule, dynamic masking policy, optimizer, scheduler, and training budget are identical. The only intended difference is the tokenizer/representation group.

The frozen lightweight character n-gram classifier is trained only from CulturaX train texts and language IDs. It supplies candidate-selection language probabilities; the trainable in-model router supplies differentiable fusion weights. Both trainers save periodic resumable checkpoints plus `best` (selected from macro-average validation MLM loss) and `final` checkpoints. In `evaluate` mode they load the selected checkpoint and report held-out CulturaX test MLM loss.

In [ ]:
from src.training.pretrain import train_from_scratch_mlm, train_language_conditioned_fused_mlm

CHECKPOINT_ROOT = PROJECT_ROOT / config['paths']['checkpoints']
LOG_ROOT = PROJECT_ROOT / config['paths']['logs']
RESULT_ROOT = PROJECT_ROOT / config['paths']['results']

bpe_model_result = train_from_scratch_mlm(
    PROJECT_ROOT, MLM_PLAN_PATH, BPE_TOKENIZER_DIR, config['model'], config['training'],
    CHECKPOINT_ROOT / 'bpe_model', 'bpe_model', config['runtime']['mode'], LOG_ROOT, RESULT_ROOT
)
probabilistic_model_result = train_language_conditioned_fused_mlm(
    PROJECT_ROOT, MLM_PLAN_PATH, PROBABILISTIC_TOKENIZER_DIR, LANGUAGE_CLASSIFIER_DIR, config['model'],
    config['probabilistic_tokenizer'], config['training'], CHECKPOINT_ROOT / 'probabilistic_model',
    config['runtime']['mode'], LOG_ROOT, RESULT_ROOT
)
print({'language_classifier': language_classifier_info, 'bpe_model': bpe_model_result, 'probabilistic_model': probabilistic_model_result})

## 9. Fine-tune the matched models on every local GLueCoS task

This stage uses the selected `best` MLM checkpoint from each group. For each fixed GLueCoS split, it runs the configured five seeds for the BPE control and the language-conditioned candidate-fusion model with the same shared task-head architecture, task hyperparameters, batch order, optimizer, linear warmup schedule, and validation-based checkpoint selection metric. The proposed arm retains candidate construction, character-span projection, router-conditioned weights, and `inputs_embeds` fusion during downstream learning.

NER, all three POS datasets, and sentiment each save per-seed predictions, metrics, epoch logs, and selected checkpoints below the configuration-controlled output paths. Existing complete per-seed results are validated and reused, making this stage restart-safe.

In [ ]:
from src.training.finetune import run_gluecos_finetuning

BPE_SELECTED_CHECKPOINT = CHECKPOINT_ROOT / 'bpe_model' / 'best'
PROBABILISTIC_SELECTED_CHECKPOINT = CHECKPOINT_ROOT / 'probabilistic_model' / 'best'
for selected_checkpoint in (BPE_SELECTED_CHECKPOINT, PROBABILISTIC_SELECTED_CHECKPOINT):
    if not (selected_checkpoint / 'trainer_state.pt').is_file():
        raise FileNotFoundError(f'Missing complete selected MLM checkpoint: {selected_checkpoint}')

finetuning_results = run_gluecos_finetuning(
    PROJECT_ROOT, config, BPE_SELECTED_CHECKPOINT, PROBABILISTIC_SELECTED_CHECKPOINT
)
print({repository: sorted(result['seeds']) for repository, result in finetuning_results.items()})

## 10. Aggregate metrics and perform paired statistical analysis

The final stage reports every seed, mean, sample standard deviation, t-based confidence interval, and paired seed-level test for accuracy, macro F1, and micro F1. It also verifies aligned BPE/proposed references and runs a paired bootstrap prediction comparison. Results are written both under `outputs/results/statistics/` and in the self-contained experiment directory.

In [ ]:
from src.evaluation.analysis import analyze_finetuning_results

statistical_report = analyze_finetuning_results(PROJECT_ROOT, config, finetuning_results)
print(json.dumps(statistical_report, indent=2, ensure_ascii=False))